# Machine Vision (UFMFRR-15-M) - Group Project
## Stage 1: Radiometric & Spatial Preprocessing (Units 2, 4, 5)

**Primary Contributor:** Jakub Wrzyszcz

This notebook applies Stage 1 preprocessing (Colour Space Decoupling, Intensity CLAHE Contrast Normalisation, and Median Spatial Denoising) to the MinneApple dataset.

In [ ]:
# Primary Contributor: Jakub Wrzyszcz

import os, glob, cv2
import matplotlib.pyplot as plt

def preprocess_stage1(rgb_img, mode="apple"):
    # 1. Colour Space Conversion (Unit 2)
    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)

    # 2. Contrast Normalisation on Intensity ONLY (Unit 4)
    clip = 2.5 if mode == "apple" else 2.0
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    v_eq = clahe.apply(v)

    # 3. Spatial Denoising (Unit 5)
    k_med = 3 if mode == "apple" else 5
    v_denoised = cv2.medianBlur(v_eq, k_med)
    v_denoised = cv2.GaussianBlur(v_denoised, (3, 3), 0.8)

    preprocessed_rgb = cv2.cvtColor(cv2.merge([h, s, v_denoised]), cv2.COLOR_HSV2RGB)
    return preprocessed_rgb, h, s, v_denoised

# Locate dataset
candidate_paths = [
    "minneapple-1-1/train/*.jpg",
    "../minneapple-1-1/train/*.jpg",
    "/content/minneapple-1-1/train/*.jpg",
    "/content/MachineVisionGroupwork/minneapple-1-1/train/*.jpg",
    "**/train/*.jpg"
]

img_files = []
for p in candidate_paths:
    matched = glob.glob(p, recursive=True)
    if matched:
        img_files = matched
        break

if not img_files:
    raise FileNotFoundError("Dataset not found! Please check minneapple-1-1/train/")

raw_rgb = cv2.cvtColor(cv2.imread(img_files[0]), cv2.COLOR_BGR2RGB)
clean_rgb, h, s, v = preprocess_stage1(raw_rgb, mode="apple")

# Display Before vs After
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1); plt.imshow(raw_rgb); plt.title(f"Original: {os.path.basename(img_files[0])}"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(clean_rgb); plt.title("Stage 1 Output (CLAHE + Median Denoised)"); plt.axis("off")
plt.tight_layout()
plt.show()
print("Stage 1 completed successfully!")
